In [ ]:
from pathlib import Path
import os
import sys
import json

REPO_ROOT = (
    Path.cwd() if (Path.cwd() / "diffusion_project").is_dir() else Path.cwd().parent
)
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# "wiener": spatial masks ranked by Wiener coefficients.
# "els": boundary-broken square patches. Model-specific minima/filenames are automatic.
EXPERIMENT = {
    "analytical_model": "wiener",
    "dataset": "celeba32",
    "data": {"root": "./data", "image_dir": "./data/celeba32", "download": False},
    "output_root": "./results",
    "subset_size": 10000,
    "num_subsets": 2,
    "device": "auto",
    "seed": 0,
    "source_batch_size": 8,
    "calibration": {
        "source_size": 5000,
        "validation_size": 2000,
        "validation_count": 2000,
        "randomize": False,
        "min_increment": 4,
        "fraction_increment": 0.05,
        "batch_size": 8,
    },
    "generation": {
        "num_samples": 100,
        "batch_size": 8,
        "method": "euler",
    },
}
# To use the supplied ELS settings: EXPERIMENT = json.loads(Path("configs/analytical_els.json").read_text())
if os.environ.get("CONFIG_FILE"):
    EXPERIMENT = json.loads(Path(os.environ["CONFIG_FILE"]).read_text())


# Analytical calibration and generation

Choose `wiener` or `els` in the first cell. Wiener LS uses filter coefficients to select
spatial support; boundary-broken ELS uses square patches with boundary-dependent sharing.
Both compute empirical-posterior denoising estimates.

Calibration saves each completed noise level. Generation uses a shared `seeds.pt` across
subsets and model families and resumes completed sample batches.
The default schedule is continuous cosine with 20 times from 0.005 to 1 and Euler integration.
Set `generation.method='ddim'` for DDIM on the same time grid.

The ELS JSON example uses two validation images; increase `validation_count` for a more
stable calibration. See [methods](../docs/methods.md) for definitions and parameter choices.


In [ ]:
from diffusion_project.analytical.pipeline import run
from diffusion_project.figures import sample_grid

outputs = run(EXPERIMENT)
if EXPERIMENT.get("generate", True):
    for subset, samples in outputs.items():
        sample_grid(samples[:8], f"{EXPERIMENT['analytical_model']} subset {subset}")


In [ ]:
import torch
import matplotlib.pyplot as plt

path = (
    Path(EXPERIMENT["output_root"])
    / "calibration"
    / EXPERIMENT["dataset"]
    / EXPERIMENT["analytical_model"]
    / "calibration.pt"
)
calibration = torch.load(path, map_location="cpu", weights_only=True)
steps = calibration["steps"]
counts = [float(torch.as_tensor(s["counts"]).float().mean()) for s in steps]
plt.plot(calibration["provenance"]["times"][: len(steps)], counts, marker="o")
plt.xlabel("Noise time")
plt.ylabel(
    "Mean selected pixels"
    if EXPERIMENT["analytical_model"] == "wiener"
    else "Patch side length"
)
plt.show()
